# Preparation

In [1]:
TESTS = [
    {
        "id": "q1",
        "question": "把 1 到 1000 中，既能被 3 整除又不能被 5 整除的整数个数算出来。只输出最终数字。",
        "gold": "267",
    },
    {
        "id": "q2",
        "question": "某商品先涨价 20%，再打 8 折，最后再涨价 25%。最终价格与原价相比是涨了、跌了还是不变？只回答“涨了X% / 跌了X% / 不变”。",
        "gold": "涨了20%",
    },
    {
        "id": "q3",
        "question": "A 比 B 大，B 比 C 大，C 不比 D 大。以下哪项一定为真？1. A 比 D 大 2. D 比 A 小 3. B 比 D 大 4. 无法确定。只输出选项编号。",
        "gold": "4",
    },
    {
        "id": "q4",
        "question": "5 个人排成一排，甲乙必须相邻，丙不能站两端。共有多少种排法？只输出数字。",
        "gold": "24",
    },
    {
        "id": "q5",
        "question": "有红、蓝、绿三种球各 4 个，从中任取 5 个。至少有 2 个同色是否必然成立？请只回答“是”或“否”，并给一句理由。",
        "gold": "是",
    },
    {
        "id": "q6",
        "question": "一个会议安排在周一到周五。已知：不在周一；如果在周三，则小王不能参加；小王必须参加；周五会议室关闭；周二主讲人出差。问会议只能安排在哪一天？只输出星期几。",
        "gold": "周四",
    },
    {
        "id": "q7",
        "question": """阅读下面 Python 代码，输出什么，只写最终输出内容：

def f(x, acc=[]):
    acc.append(x)
    return acc

print(f(1))
print(f(2))
print(f(3, []))
print(f(4))
""",
        "gold": "[1]\n[1, 2]\n[3]\n[1, 2, 4]",
    },
    {
        "id": "q8",
        "question": """下面代码的时间复杂度是多少？只回答 O(...)。

count = 0
for i in range(n):
    j = 1
    while j < n:
        count += 1
        j *= 2
""",
        "gold": "O(n log n)",
    },
    {
        "id": "q9",
        "question": """下面函数想返回列表中第二大的不同元素，但有 bug。请指出 bug 本质，只用一句话回答。

def second_largest(nums):
    nums.sort()
    return nums[-2]
""",
        "gold": "没有处理重复值",
    },
    {
        "id": "q10",
        "question": "一辆车从 A 到 B，去程速度 60，回程速度 40。求往返平均速度。只输出数字，不要解释。",
        "gold": "48",
    },
    {
        "id": "q11",
        "question": "如果今天是星期二，那么从今天起第 100 天是星期几？只输出星期几。",
        "gold": "星期四",
    },
    {
        "id": "q12",
        "question": "有 8 升和 5 升两个空壶，以及无限水源。请问是否能精确量出 1 升？只回答“能/不能”，再给最短步骤。",
        "gold": "能",
    },
]

In [ ]:
def build_prompt(question: str) -> str:
    return f"""你将回答一道推理题。

要求：
1. 不要输出思考过程。
2. 如果得出答案，请立刻结束。
3. 最后一行必须输出一个 JSON 对象。
4. JSON 格式必须是：{{"final_answer":"你的答案"}}
5. 不要在最后一行输出其他内容。

题目如下：
{question}
"""


In [ ]:
import gc
import importlib
import json
import re
import time
from pathlib import Path

import pandas as pd


In [4]:
def extract_final_answer_json(text: str) -> str:
    lines = [x.strip() for x in text.splitlines() if x.strip()]
    for line in reversed(lines):
        try:
            obj = json.loads(line)
            if isinstance(obj, dict) and "final_answer" in obj:
                return str(obj["final_answer"]).strip()
        except Exception:
            pass

    m = re.findall(r'(?im)^\s*FINAL\s*:\s*(.+?)\s*$', text)
    if m:
        return m[-1].strip()

    return lines[-1] if lines else ""

In [ ]:
def release_model_resources(model=None, processor=None):
    del model
    del processor
    gc.collect()

    try:
        import mlx.core as mx

        if hasattr(mx, "clear_cache"):
            mx.clear_cache()
        if hasattr(mx, "metal") and hasattr(mx.metal, "clear_cache"):
            mx.metal.clear_cache()
    except Exception:
        pass


def get_backend_bundle(backend: str):
    if backend == "mlx_lm":
        module = importlib.import_module("mlx_lm")
        return {
            "load": module.load,
            "generate": module.generate,
        }

    if backend == "mlx_vlm":
        module = importlib.import_module("mlx_vlm")
        prompt_utils = importlib.import_module("mlx_vlm.prompt_utils")
        utils = importlib.import_module("mlx_vlm.utils")
        return {
            "load": module.load,
            "generate": module.generate,
            "apply_chat_template": prompt_utils.apply_chat_template,
            "load_config": utils.load_config,
        }

    raise ValueError(f"Unsupported backend: {backend}")


def build_generation_prompt(backend: str, processor, config, messages, enable_thinking: bool):
    if backend == "mlx_lm":
        return processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_dict=False,
            enable_thinking=enable_thinking,
        )

    if backend == "mlx_vlm":
        return get_backend_bundle(backend)["apply_chat_template"](
            processor,
            config,
            messages,
            add_generation_prompt=True,
            num_images=0,
            enable_thinking=enable_thinking,
        )

    raise ValueError(f"Unsupported backend: {backend}")


def run_one_model(
    model_name: str,
    backend: str,
    tests,
    max_tokens: int = 512,
    enable_thinking: bool = False,
):
    backend_bundle = get_backend_bundle(backend)
    model = None
    processor = None
    config = None

    print(f"Loading model: {model_name}")
    print(f"Backend: {backend}")
    model, processor = backend_bundle["load"](model_name)
    if backend == "mlx_vlm":
        config = backend_bundle["load_config"](model_name)
    print("Model loaded.\n")

    results = []

    try:
        for i, item in enumerate(tests, 1):
            prompt_text = build_prompt(item["question"])
            messages = [{"role": "user", "content": prompt_text}]
            prompt = build_generation_prompt(
                backend=backend,
                processor=processor,
                config=config,
                messages=messages,
                enable_thinking=enable_thinking,
            )

            print(f"[{i}/{len(tests)}] Running {item['id']} ...")
            t0 = time.time()
            if backend == "mlx_lm":
                out = backend_bundle["generate"](
                    model,
                    processor,
                    prompt=prompt,
                    max_tokens=max_tokens,
                    verbose=False,
                )
            else:
                out = backend_bundle["generate"](
                    model,
                    processor,
                    prompt=prompt,
                    max_tokens=max_tokens,
                    verbose=False,
                    enable_thinking=enable_thinking,
                )
            latency = time.time() - t0

            text = out.text if hasattr(out, "text") else str(out)
            text = text.strip()
            final_pred = extract_final_answer_json(text)

            results.append({
                "model": model_name,
                "backend": backend,
                "qid": item["id"],
                "question": item["question"],
                "gold": item["gold"],
                "pred": text,
                "final_pred": final_pred,
                "latency_sec": round(latency, 2),
                "pred_len": len(text),
                "final_pred_len": len(final_pred),
            })
    finally:
        release_model_resources(model=model, processor=processor)
        print("Model released.\n")

    return results


# Configure Comparison Pair

选择 `ACTIVE_COMPARISON` 来切换对比组。

内存有限时建议按这个顺序执行：
1. 先运行配置 cell。
2. 跑 `MODEL_A` 并保存结果。
3. 如果还担心内存占用，重启 kernel。
4. 重新执行 Preparation 和配置 cell，再跑 `MODEL_B` 并保存结果。
5. 最后直接从磁盘加载两个结果文件做比较。


In [ ]:
COMPARISON_PRESETS = {
    "qwen": {
        "label": "Qwen 3.5",
        "model_a": "mlx-community/Qwen3.5-27B-Claude-4.6-Opus-Distilled-MLX-4bit",
        "model_b": "mlx-community/Qwen3.5-35B-A3B-4bit",
        "backend_a": "mlx_lm",
        "backend_b": "mlx_lm",
        "max_tokens_a": 2048,
        "max_tokens_b": 256,
    },
    "gemma": {
        "label": "Gemma 4",
        "model_a": "mlx-community/gemma-4-26b-a4b-it-4bit",
        "model_b": "mlx-community/gemma-4-31b-it-4bit",
        "backend_a": "mlx_vlm",
        "backend_b": "mlx_vlm",
        "max_tokens_a": 256,
        "max_tokens_b": 256,
    },
}

ACTIVE_COMPARISON = "gemma"
config = COMPARISON_PRESETS[ACTIVE_COMPARISON]

RESULTS_DIR = Path("benchmark_results")
RESULTS_DIR.mkdir(exist_ok=True)
RESULTS_A_PATH = RESULTS_DIR / f"{ACTIVE_COMPARISON}_results_model_a.json"
RESULTS_B_PATH = RESULTS_DIR / f"{ACTIVE_COMPARISON}_results_model_b.json"

MODEL_A = config["model_a"]
MODEL_B = config["model_b"]
BACKEND_A = config["backend_a"]
BACKEND_B = config["backend_b"]
MAX_TOKENS_A = config["max_tokens_a"]
MAX_TOKENS_B = config["max_tokens_b"]
ENABLE_THINKING = False

print(f"Active comparison: {config['label']} ({ACTIVE_COMPARISON})")
print(f"MODEL_A: {MODEL_A} [{BACKEND_A}]")
print(f"MODEL_B: {MODEL_B} [{BACKEND_B}]")
print(f"Results A path: {RESULTS_A_PATH}")
print(f"Results B path: {RESULTS_B_PATH}")
print(f"enable_thinking={ENABLE_THINKING}")


In [ ]:
results_a = run_one_model(
    model_name=MODEL_A,
    backend=BACKEND_A,
    tests=TESTS,
    max_tokens=MAX_TOKENS_A,
    enable_thinking=ENABLE_THINKING,
)

with open(RESULTS_A_PATH, "w", encoding="utf-8") as f:
    json.dump(results_a, f, ensure_ascii=False, indent=2)

pd.DataFrame(results_a)


# Run Model B


In [ ]:
results_b = run_one_model(
    model_name=MODEL_B,
    backend=BACKEND_B,
    tests=TESTS,
    max_tokens=MAX_TOKENS_B,
    enable_thinking=ENABLE_THINKING,
)


In [ ]:
with open(RESULTS_B_PATH, "w", encoding="utf-8") as f:
    json.dump(results_b, f, ensure_ascii=False, indent=2)

pd.DataFrame(results_b)


# Comparison


In [ ]:
with open(RESULTS_A_PATH, "r", encoding="utf-8") as f:
    results_a = json.load(f)

with open(RESULTS_B_PATH, "r", encoding="utf-8") as f:
    results_b = json.load(f)

df_a = pd.DataFrame(results_a)
df_b = pd.DataFrame(results_b)

df_a[["qid", "gold", "final_pred", "latency_sec", "final_pred_len"]]


In [9]:
def normalize_text(s: str) -> str:
    s = s.strip()
    s = s.replace("（", "(").replace("）", ")")
    s = s.replace("：", ":")
    s = re.sub(r"\s+", " ", s)
    return s

def score_row(qid: str, gold: str, pred: str) -> int:
    g = normalize_text(gold)
    p = normalize_text(pred)

    # 精确类题目
    exact_match_ids = {"q1", "q2", "q3", "q4", "q6", "q8", "q10", "q11"}
    if qid in exact_match_ids:
        return int(g == p)

    # 包含关键短语即可
    if qid == "q5":
        return int("是" in p)

    if qid == "q9":
        keys = ["重复", "第二大不同", "没有处理重复值"]
        return int(any(k in p for k in keys))

    if qid == "q12":
        return int("能" in p)

    if qid == "q7":
        # 对代码输出题做宽松匹配
        want_parts = ["[1]", "[1, 2]", "[3]", "[1, 2, 4]"]
        return int(all(x in pred for x in want_parts))

    return 0

df_a["correct"] = df_a.apply(lambda r: score_row(r["qid"], r["gold"], r["final_pred"]), axis=1)
df_b["correct"] = df_b.apply(lambda r: score_row(r["qid"], r["gold"], r["final_pred"]), axis=1)

In [10]:
summary = pd.DataFrame([
    {
        "model": df_a["model"].iloc[0],
        "correct_count": int(df_a["correct"].sum()),
        "avg_latency_sec": round(df_a["latency_sec"].mean(), 2),
        "avg_final_pred_len": round(df_a["final_pred_len"].mean(), 1),
    },
    {
        "model": df_b["model"].iloc[0],
        "correct_count": int(df_b["correct"].sum()),
        "avg_latency_sec": round(df_b["latency_sec"].mean(), 2),
        "avg_final_pred_len": round(df_b["final_pred_len"].mean(), 1),
    }
])

summary

,model,correct_count,avg_latency_sec,avg_final_pred_len
0,mlx-community/Qwen3.5-27B-Claude-4.6-Opus-Dist...,12,20.18,8.2
1,mlx-community/Qwen3.5-35B-A3B-4bit,10,0.71,6.2


In [11]:
compare = df_a[["qid", "gold", "final_pred", "correct", "latency_sec"]].rename(columns={
    "final_pred": "final_pred_a",
    "correct": "correct_a",
    "latency_sec": "latency_a",
}).merge(
    df_b[["qid", "final_pred", "correct", "latency_sec"]].rename(columns={
        "final_pred": "final_pred_b",
        "correct": "correct_b",
        "latency_sec": "latency_b",
    }),
    on="qid",
)

compare

,qid,gold,final_pred_a,correct_a,latency_a,final_pred_b,correct_b,latency_b
0,q1,267,267,1,25.29,267,1,2.20
1,q2,涨了20%,涨了20%,1,32.14,涨了20%,1,1.42
2,q3,4,4,1,20.67,4,1,0.32
3,q4,24,24,1,40.29,18,0,0.27
4,q5,是,是,1,13.97,是,1,0.66
5,q6,周四,周四,1,15.19,周四,1,0.36
6,q7,"[1]\n[1, 2]\n[3]\n[1, 2, 4]","[1]\n[1, 2]\n[3]\n[1, 2, 4]",1,17.08,"[1]\n[1, 2]\n[4]\n[1, 2, 4]",0,0.69
7,q8,O(n log n),O(n log n),1,8.21,O(n log n),1,0.39
8,q9,没有处理重复值,该函数没有先去重，当列表中存在重复的最大值时，会错误地返回最大值本身而非第二大的不同元素。,1,18.91,未处理重复元素导致第二大不同元素缺失,1,0.50
9,q10,48,48,1,7.92,48,1,0.34
